# Coastal flood step 03: post-processing and summaries (maximum_scenario)

Builds grouped damage layers and summary tables for this set scenario.


In [ ]:
import geopandas
import numpy
import pandas
from pathlib import Path

base_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers")


In [ ]:
output_path = base_path / "dphil_paper_3/results/02_damage_estimates/coastal_flood_damages/results_coastal_maximum_scenario"
data_root = base_path / "dphil_common_cross_cutting/common_incoming_data"
intersections_results_directory = base_path / "dphil_paper_3/results/01_hazard_infrastructure_network_intersections/coastal_flood_network_intersections"
reprojected_networks_root = intersections_results_directory / "inputs_reprojected_jamaica_crs/networks"
network_csv = data_root / "networks/network_layers_hazard_intersections_details.csv"
jamaica_crs = 3448

damage_results_directory = output_path / "direct_damages"
damage_estimates_directory = output_path / "damage_estimates"
damage_estimates_directory.mkdir(parents=True, exist_ok=True)

if not reprojected_networks_root.exists():
    raise FileNotFoundError(f"Missing reprojected network folder: {reprojected_networks_root}")

network_data_table = pandas.read_csv(network_csv)


In [ ]:
mangrove_flood_damage_columns = [
    "coastal_flood_fn_mg_rp_25",
    "coastal_flood_fn_mg_rp_100",
    "coastal_flood_fn_mg_rp_500",
]
nomangrove_flood_damage_columns = [
    "coastal_flood_fn_nomg_rp_25",
    "coastal_flood_fn_nomg_rp_100",
    "coastal_flood_fn_nomg_rp_500",
]
difference_columns = [
    "coastal_flood_diff_rp_25",
    "coastal_flood_diff_rp_100",
    "coastal_flood_diff_rp_500",
]
flood_damage_columns = mangrove_flood_damage_columns + nomangrove_flood_damage_columns + difference_columns


In [ ]:
damage_totals = []
for asset_info in network_data_table.itertuples(index=False):
    asset_gpkg = asset_info.asset_gpkg
    asset_layer = asset_info.asset_layer
    asset_id = asset_info.asset_id_column
    damage_file = (
        damage_results_directory
        / f"{asset_gpkg}_{asset_layer}"
        / f"{asset_gpkg}_{asset_layer}_direct_damages_parameter_set_0.parquet"
    )

    if not damage_file.exists():
        continue

    damage_table = pandas.read_parquet(damage_file)
    damage_table[difference_columns] = damage_table[nomangrove_flood_damage_columns] - damage_table[mangrove_flood_damage_columns].values
    damage_uncertainty_parameter = damage_table["damage_uncertainty_parameter"].iloc[0] if ("damage_uncertainty_parameter" in damage_table.columns and not damage_table.empty) else numpy.nan
    cost_uncertainty_parameter = damage_table["cost_uncertainty_parameter"].iloc[0] if ("cost_uncertainty_parameter" in damage_table.columns and not damage_table.empty) else numpy.nan

    grouped_damages = damage_table.groupby([asset_id], as_index=False)[flood_damage_columns].sum()
    asset_relative_path = Path(asset_info.path)
    if asset_relative_path.parts and asset_relative_path.parts[0] == "networks":
        asset_relative_path = Path(*asset_relative_path.parts[1:])
    asset_file = reprojected_networks_root / asset_relative_path
    if not asset_file.exists():
        raise FileNotFoundError(f"Could not find reprojected asset file: {asset_file}")

    asset_geometry = geopandas.read_file(asset_file, layer=asset_layer)
    asset_geometry = asset_geometry.to_crs(epsg=jamaica_crs)
    grouped_damages = pandas.merge(grouped_damages, asset_geometry[[asset_id, "geometry"]], how="left", on=[asset_id])
    grouped_damages = geopandas.GeoDataFrame(grouped_damages, geometry="geometry", crs=jamaica_crs)

    grouped_damage_file = damage_estimates_directory / f"{asset_gpkg}_{asset_layer}_asset_damages_groupedby.gpkg"
    grouped_damages.to_file(grouped_damage_file, driver="GPKG")

    grouped_damages["sector"] = asset_gpkg
    grouped_damages["layer"] = asset_layer
    grouped_damages["damage_uncertainty_parameter"] = damage_uncertainty_parameter
    grouped_damages["cost_uncertainty_parameter"] = cost_uncertainty_parameter

    aggregate_columns = {column_name: "sum" for column_name in flood_damage_columns}
    aggregate_columns["damage_uncertainty_parameter"] = "first"
    aggregate_columns["cost_uncertainty_parameter"] = "first"
    grouped_damages = grouped_damages.groupby(["sector", "layer"], as_index=False).agg(aggregate_columns)
    damage_totals.append(grouped_damages)

if damage_totals:
    damage_totals = pandas.concat(damage_totals, axis=0, ignore_index=True)
else:
    damage_totals = pandas.DataFrame(columns=["sector", "layer"] + flood_damage_columns + ["damage_uncertainty_parameter", "cost_uncertainty_parameter"])

damage_totals.to_csv(damage_estimates_directory / "asset_damages_groupedby.csv", index=False)
print(f"Saved: {damage_estimates_directory / 'asset_damages_groupedby.csv'}")


In [ ]:
if not damage_estimates_directory.exists():
    raise FileNotFoundError(f"Missing folder: {damage_estimates_directory}")

damage_estimate_files = sorted(damage_estimates_directory.glob("*_asset_damages_groupedby.gpkg"))
if not damage_estimate_files:
    raise FileNotFoundError(f"No grouped damage GPKGs found in {damage_estimates_directory}")

summary_rows = []
for grouped_damage_file in damage_estimate_files:
    grouped_damage = geopandas.read_file(grouped_damage_file)
    asset_name = grouped_damage_file.stem.replace("_asset_damages_groupedby", "")
    summary_rows.append({
        "asset_name": asset_name,
        "row_count": len(grouped_damage),
        "column_count": len(grouped_damage.columns),
    })

pandas.DataFrame(summary_rows).sort_values("asset_name").reset_index(drop=True)


## Post-processing (clean and reproducible)
The cells below replace old duplicate/debug cells and give reproducible outputs.

In [ ]:
# Choose one output to inspect (this reproduces table outputs like the old debug cells).
asset_name_to_preview = "pipelines_NWC_edges"  # e.g. rail_nodes, roads_edges, buildings_assigned_economic_activity_areas
preview_file = damage_estimates_directory / f"{asset_name_to_preview}_asset_damages_groupedby.gpkg"

if not preview_file.exists():
    raise FileNotFoundError(f"Missing file: {preview_file}")

preview_table = geopandas.read_file(preview_file)
print(f"Rows: {len(preview_table)} | Columns: {len(preview_table.columns)}")
preview_table.head(20)


In [ ]:
# Sum damages by Sector, Subsector, and ReturnPeriod.
# Avoided damages are signed: Without_Mangroves - With_Mangroves (can be negative).
# Readable tables are shown in USD using 1 USD = 150 JD.
network_details = network_data_table[["sector", "asset_description", "asset_gpkg", "asset_layer"]].drop_duplicates()
summary_value_columns_jd = [
    "Damages_With_Mangroves_JD",
    "Damages_Without_Mangroves_JD",
    "Avoided_Damages_JD",
    "NomgMinusMg_Raster_Damages_JD",
]
usd_exchange_rate_jd_per_usd = 150.0

damage_rows = []
for asset_info in network_details.itertuples(index=False):
    damage_file = (
        damage_results_directory
        / f"{asset_info.asset_gpkg}_{asset_info.asset_layer}"
        / f"{asset_info.asset_gpkg}_{asset_info.asset_layer}_direct_damages_parameter_set_0.parquet"
    )

    if not damage_file.exists():
        continue

    damage_table = pandas.read_parquet(damage_file)
    return_periods = sorted({column_name.split("_rp_")[-1] for column_name in damage_table.columns if "_rp_" in column_name})

    for return_period in return_periods:
        damages_with_mangroves_column = f"coastal_flood_fn_mg_rp_{return_period}"
        damages_without_mangroves_column = f"coastal_flood_fn_nomg_rp_{return_period}"
        nomangrove_minus_mangrove_column = f"coastal_flood_fn_nomg_minus_mg_rp_{return_period}"

        damages_with_mangroves = (
            float(damage_table[damages_with_mangroves_column].sum())
            if damages_with_mangroves_column in damage_table.columns
            else 0.0
        )

        if damages_without_mangroves_column in damage_table.columns:
            damages_without_mangroves = float(damage_table[damages_without_mangroves_column].sum())
        elif nomangrove_minus_mangrove_column in damage_table.columns:
            damages_without_mangroves = (
                damages_with_mangroves + float(damage_table[nomangrove_minus_mangrove_column].sum())
            )
        else:
            damages_without_mangroves = 0.0

        avoided_damages = damages_without_mangroves - damages_with_mangroves
        nomangrove_minus_mangrove_raster_damages = (
            float(damage_table[nomangrove_minus_mangrove_column].sum())
            if nomangrove_minus_mangrove_column in damage_table.columns
            else float("nan")
        )

        damage_rows.append({
            "Sector": asset_info.sector,
            "Subsector": asset_info.asset_description,
            "Asset": asset_info.asset_gpkg,
            "Layer": asset_info.asset_layer,
            "ReturnPeriod": int(return_period),
            "Damages_With_Mangroves_JD": damages_with_mangroves,
            "Damages_Without_Mangroves_JD": damages_without_mangroves,
            "Avoided_Damages_JD": avoided_damages,
            "NomgMinusMg_Raster_Damages_JD": nomangrove_minus_mangrove_raster_damages,
        })

asset_level_summary = pandas.DataFrame(damage_rows)

sector_subsector_summary_jd = (
    asset_level_summary
    .groupby(["Sector", "Subsector", "ReturnPeriod"], as_index=False)[summary_value_columns_jd]
    .sum()
)

sector_summary_jd = (
    sector_subsector_summary_jd
    .groupby(["Sector", "ReturnPeriod"], as_index=False)[summary_value_columns_jd]
    .sum()
)

total_return_period_summary_jd = (
    sector_summary_jd
    .groupby(["ReturnPeriod"], as_index=False)[summary_value_columns_jd]
    .sum()
)


def add_damage_metrics(summary_table, damages_without_column, avoided_damages_column):
    summary_table = summary_table.copy()
    summary_table["Reduction_percent"] = numpy.where(
        summary_table[damages_without_column] != 0,
        100.0 * summary_table[avoided_damages_column] / summary_table[damages_without_column],
        numpy.nan,
    )
    return summary_table


def convert_jd_summary_to_usd(summary_table):
    usd_summary_table = summary_table.copy()
    jd_value_columns = [column_name for column_name in usd_summary_table.columns if column_name.endswith("_JD")]

    for column_name in jd_value_columns:
        usd_column_name = column_name.replace("_JD", "_USD")
        usd_summary_table[usd_column_name] = usd_summary_table[column_name] / usd_exchange_rate_jd_per_usd

    usd_summary_table = usd_summary_table.drop(columns=jd_value_columns)
    return usd_summary_table


def format_us_dollars(value):
    if pandas.isna(value):
        return ""

    absolute_value = abs(value)
    sign = "-" if value < 0 else ""

    if absolute_value >= 1_000_000_000:
        return f"{sign}US${absolute_value / 1_000_000_000:,.2f} billion"
    if absolute_value >= 1_000_000:
        return f"{sign}US${absolute_value / 1_000_000:,.2f} million"
    if absolute_value >= 1_000:
        return f"{sign}US${absolute_value / 1_000:,.1f} thousand"
    return f"{sign}US${absolute_value:,.0f}"


def format_percent(value):
    if pandas.isna(value):
        return ""
    return f"{value:,.1f}%"


def build_readable_usd_table(summary_table, group_columns, include_total_share):
    readable_table = summary_table[group_columns].copy()
    readable_table["Damages With Mangroves"] = summary_table["Damages_With_Mangroves_USD"].apply(format_us_dollars)
    readable_table["Damages Without Mangroves"] = summary_table["Damages_Without_Mangroves_USD"].apply(format_us_dollars)
    readable_table["Avoided Damages"] = summary_table["Avoided_Damages_USD"].apply(format_us_dollars)
    readable_table["Reduction"] = summary_table["Reduction_percent"].apply(format_percent)

    if include_total_share:
        readable_table["Share of Total Avoided"] = summary_table["Share_of_Total_Avoided_percent"].apply(format_percent)

    return readable_table


total_return_period_summary_jd = add_damage_metrics(
    total_return_period_summary_jd,
    damages_without_column="Damages_Without_Mangroves_JD",
    avoided_damages_column="Avoided_Damages_JD",
)
sector_summary_jd = add_damage_metrics(
    sector_summary_jd,
    damages_without_column="Damages_Without_Mangroves_JD",
    avoided_damages_column="Avoided_Damages_JD",
)
sector_subsector_summary_jd = add_damage_metrics(
    sector_subsector_summary_jd,
    damages_without_column="Damages_Without_Mangroves_JD",
    avoided_damages_column="Avoided_Damages_JD",
)

total_avoided_by_return_period_jd = total_return_period_summary_jd[["ReturnPeriod", "Avoided_Damages_JD"]].rename(
    columns={"Avoided_Damages_JD": "Total_Avoided_Damages_JD"}
)

sector_summary_jd = sector_summary_jd.merge(total_avoided_by_return_period_jd, on="ReturnPeriod", how="left")
sector_subsector_summary_jd = sector_subsector_summary_jd.merge(total_avoided_by_return_period_jd, on="ReturnPeriod", how="left")

sector_summary_jd["Share_of_Total_Avoided_percent"] = numpy.where(
    sector_summary_jd["Total_Avoided_Damages_JD"] != 0,
    100.0 * sector_summary_jd["Avoided_Damages_JD"] / sector_summary_jd["Total_Avoided_Damages_JD"],
    numpy.nan,
)
sector_subsector_summary_jd["Share_of_Total_Avoided_percent"] = numpy.where(
    sector_subsector_summary_jd["Total_Avoided_Damages_JD"] != 0,
    100.0 * sector_subsector_summary_jd["Avoided_Damages_JD"] / sector_subsector_summary_jd["Total_Avoided_Damages_JD"],
    numpy.nan,
)

total_return_period_summary_jd = total_return_period_summary_jd.sort_values(["ReturnPeriod"]).reset_index(drop=True)
sector_summary_jd = sector_summary_jd.sort_values(
    ["ReturnPeriod", "Avoided_Damages_JD", "Sector"],
    ascending=[True, False, True],
).reset_index(drop=True)
sector_subsector_summary_jd = sector_subsector_summary_jd.sort_values(
    ["ReturnPeriod", "Sector", "Avoided_Damages_JD", "Subsector"],
    ascending=[True, True, False, True],
).reset_index(drop=True)

total_return_period_summary_usd = convert_jd_summary_to_usd(total_return_period_summary_jd)
sector_summary_usd = convert_jd_summary_to_usd(sector_summary_jd)
sector_subsector_summary_usd = convert_jd_summary_to_usd(sector_subsector_summary_jd)

total_return_period_summary_readable_usd = build_readable_usd_table(
    total_return_period_summary_usd,
    group_columns=["ReturnPeriod"],
    include_total_share=False,
)
sector_summary_readable_usd = build_readable_usd_table(
    sector_summary_usd,
    group_columns=["ReturnPeriod", "Sector"],
    include_total_share=True,
)
sector_subsector_summary_readable_usd = build_readable_usd_table(
    sector_subsector_summary_usd,
    group_columns=["ReturnPeriod", "Sector", "Subsector"],
    include_total_share=True,
)

total_return_period_summary_file_jd = damage_estimates_directory / "total_return_period_damages.csv"
sector_summary_file_jd = damage_estimates_directory / "sector_return_period_damages.csv"
sector_subsector_summary_file_jd = damage_estimates_directory / "sector_subsector_return_period_damages.csv"

total_return_period_summary_file_usd = damage_estimates_directory / "total_return_period_damages_usd.csv"
sector_summary_file_usd = damage_estimates_directory / "sector_return_period_damages_usd.csv"
sector_subsector_summary_file_usd = damage_estimates_directory / "sector_subsector_return_period_damages_usd.csv"

total_return_period_summary_readable_file_usd = damage_estimates_directory / "total_return_period_damages_readable_usd.csv"
sector_summary_readable_file_usd = damage_estimates_directory / "sector_return_period_damages_readable_usd.csv"
sector_subsector_summary_readable_file_usd = damage_estimates_directory / "sector_subsector_return_period_damages_readable_usd.csv"

total_return_period_summary_jd.to_csv(total_return_period_summary_file_jd, index=False)
sector_summary_jd.to_csv(sector_summary_file_jd, index=False)
sector_subsector_summary_jd.to_csv(sector_subsector_summary_file_jd, index=False)

total_return_period_summary_usd.to_csv(total_return_period_summary_file_usd, index=False)
sector_summary_usd.to_csv(sector_summary_file_usd, index=False)
sector_subsector_summary_usd.to_csv(sector_subsector_summary_file_usd, index=False)

total_return_period_summary_readable_usd.to_csv(total_return_period_summary_readable_file_usd, index=False)
sector_summary_readable_usd.to_csv(sector_summary_readable_file_usd, index=False)
sector_subsector_summary_readable_usd.to_csv(sector_subsector_summary_readable_file_usd, index=False)

print(f"Assumed exchange rate: 1 USD = {usd_exchange_rate_jd_per_usd:,.0f} JD")
print(f"Saved: {total_return_period_summary_file_jd}")
print(f"Saved: {sector_summary_file_jd}")
print(f"Saved: {sector_subsector_summary_file_jd}")
print(f"Saved: {total_return_period_summary_file_usd}")
print(f"Saved: {sector_summary_file_usd}")
print(f"Saved: {sector_subsector_summary_file_usd}")
print(f"Saved: {total_return_period_summary_readable_file_usd}")
print(f"Saved: {sector_summary_readable_file_usd}")
print(f"Saved: {sector_subsector_summary_readable_file_usd}")
print()
print("Readable total damages by return period (USD):")
display(total_return_period_summary_readable_usd)
print()
print("Readable sector damages by return period (USD):")
display(sector_summary_readable_usd)
print()
print("Readable sector + subsector damages by return period (USD):")
display(sector_subsector_summary_readable_usd)

print()
positive_avoided_by_return_period_usd = (
    sector_subsector_summary_usd
    .assign(Positive_Avoided_Damages_USD=sector_subsector_summary_usd["Avoided_Damages_USD"].clip(lower=0))
    .groupby("ReturnPeriod", as_index=False)["Positive_Avoided_Damages_USD"]
    .sum()
)

damage_increase_subsector_summary_usd = sector_subsector_summary_usd[
    sector_subsector_summary_usd["Avoided_Damages_USD"] < 0
].copy()
damage_increase_subsector_summary_usd["Damage_Increase_USD"] = -damage_increase_subsector_summary_usd["Avoided_Damages_USD"]
damage_increase_subsector_summary_usd["Damage_Increase_percent_of_NoMangrove_Damages"] = numpy.where(
    damage_increase_subsector_summary_usd["Damages_Without_Mangroves_USD"] != 0,
    100.0
    * damage_increase_subsector_summary_usd["Damage_Increase_USD"]
    / damage_increase_subsector_summary_usd["Damages_Without_Mangroves_USD"],
    numpy.nan,
)
damage_increase_subsector_summary_usd["Damage_Increase_share_of_Net_Avoided_percent"] = numpy.where(
    damage_increase_subsector_summary_usd["Total_Avoided_Damages_USD"] != 0,
    100.0
    * damage_increase_subsector_summary_usd["Damage_Increase_USD"]
    / damage_increase_subsector_summary_usd["Total_Avoided_Damages_USD"],
    numpy.nan,
)

damage_increase_by_return_period_usd = (
    damage_increase_subsector_summary_usd
    .groupby("ReturnPeriod", as_index=False)["Damage_Increase_USD"]
    .sum()
)

damage_increase_return_period_summary_usd = (
    total_return_period_summary_usd[["ReturnPeriod", "Avoided_Damages_USD"]]
    .rename(columns={"Avoided_Damages_USD": "Net_Avoided_Damages_USD"})
    .merge(damage_increase_by_return_period_usd, on="ReturnPeriod", how="left")
    .merge(positive_avoided_by_return_period_usd, on="ReturnPeriod", how="left")
)
damage_increase_return_period_summary_usd["Damage_Increase_USD"] = damage_increase_return_period_summary_usd["Damage_Increase_USD"].fillna(0.0)
damage_increase_return_period_summary_usd["Positive_Avoided_Damages_USD"] = damage_increase_return_period_summary_usd["Positive_Avoided_Damages_USD"].fillna(0.0)
damage_increase_return_period_summary_usd["Gross_Local_Effects_USD"] = (
    damage_increase_return_period_summary_usd["Positive_Avoided_Damages_USD"]
    + damage_increase_return_period_summary_usd["Damage_Increase_USD"]
)
damage_increase_return_period_summary_usd["Damage_Increase_share_of_Net_Avoided_percent"] = numpy.where(
    damage_increase_return_period_summary_usd["Net_Avoided_Damages_USD"] != 0,
    100.0
    * damage_increase_return_period_summary_usd["Damage_Increase_USD"]
    / damage_increase_return_period_summary_usd["Net_Avoided_Damages_USD"],
    numpy.nan,
)
damage_increase_return_period_summary_usd["Damage_Increase_share_of_Gross_Local_Effects_percent"] = numpy.where(
    damage_increase_return_period_summary_usd["Gross_Local_Effects_USD"] != 0,
    100.0
    * damage_increase_return_period_summary_usd["Damage_Increase_USD"]
    / damage_increase_return_period_summary_usd["Gross_Local_Effects_USD"],
    numpy.nan,
)

affected_subsector_counts = (
    damage_increase_subsector_summary_usd
    .groupby("ReturnPeriod", as_index=False)
    .size()
    .rename(columns={"size": "Affected_Subsector_Count"})
)
damage_increase_return_period_summary_usd = damage_increase_return_period_summary_usd.merge(
    affected_subsector_counts,
    on="ReturnPeriod",
    how="left",
)
damage_increase_return_period_summary_usd["Affected_Subsector_Count"] = (
    damage_increase_return_period_summary_usd["Affected_Subsector_Count"].fillna(0).astype(int)
)

damage_increase_by_return_period_totals = (
    damage_increase_subsector_summary_usd
    .groupby("ReturnPeriod", as_index=False)["Damage_Increase_USD"]
    .sum()
    .rename(columns={"Damage_Increase_USD": "ReturnPeriod_Damage_Increase_USD"})
)
damage_increase_subsector_summary_usd = damage_increase_subsector_summary_usd.merge(
    damage_increase_by_return_period_totals,
    on="ReturnPeriod",
    how="left",
)
damage_increase_subsector_summary_usd["Share_of_ReturnPeriod_Damage_Increase_percent"] = numpy.where(
    damage_increase_subsector_summary_usd["ReturnPeriod_Damage_Increase_USD"] != 0,
    100.0
    * damage_increase_subsector_summary_usd["Damage_Increase_USD"]
    / damage_increase_subsector_summary_usd["ReturnPeriod_Damage_Increase_USD"],
    numpy.nan,
)

damage_increase_return_period_summary_usd = damage_increase_return_period_summary_usd.sort_values(
    ["ReturnPeriod"]
).reset_index(drop=True)
damage_increase_subsector_summary_usd = damage_increase_subsector_summary_usd.sort_values(
    ["ReturnPeriod", "Damage_Increase_USD", "Sector", "Subsector"],
    ascending=[True, False, True, True],
).reset_index(drop=True)

damage_increase_return_period_summary_readable_usd = damage_increase_return_period_summary_usd[[
    "ReturnPeriod"
]].copy()
damage_increase_return_period_summary_readable_usd["Total Damage Increase"] = damage_increase_return_period_summary_usd[
    "Damage_Increase_USD"
].apply(format_us_dollars)
damage_increase_return_period_summary_readable_usd["Share of Net Avoided"] = damage_increase_return_period_summary_usd[
    "Damage_Increase_share_of_Net_Avoided_percent"
].apply(format_percent)
damage_increase_return_period_summary_readable_usd["Share of Gross Local Effects"] = damage_increase_return_period_summary_usd[
    "Damage_Increase_share_of_Gross_Local_Effects_percent"
].apply(format_percent)
damage_increase_return_period_summary_readable_usd["Affected Subsectors"] = damage_increase_return_period_summary_usd[
    "Affected_Subsector_Count"
]

damage_increase_subsector_summary_readable_usd = damage_increase_subsector_summary_usd[[
    "ReturnPeriod", "Sector", "Subsector"
]].copy()
damage_increase_subsector_summary_readable_usd["Damage Increase"] = damage_increase_subsector_summary_usd[
    "Damage_Increase_USD"
].apply(format_us_dollars)
damage_increase_subsector_summary_readable_usd["Increase vs No-Mangrove Baseline"] = damage_increase_subsector_summary_usd[
    "Damage_Increase_percent_of_NoMangrove_Damages"
].apply(format_percent)
damage_increase_subsector_summary_readable_usd["Share of Net Avoided"] = damage_increase_subsector_summary_usd[
    "Damage_Increase_share_of_Net_Avoided_percent"
].apply(format_percent)
damage_increase_subsector_summary_readable_usd["Share of Return-Period Damage Increase"] = damage_increase_subsector_summary_usd[
    "Share_of_ReturnPeriod_Damage_Increase_percent"
].apply(format_percent)

damage_increase_return_period_summary_file_usd = (
    damage_estimates_directory / "damage_increase_return_period_summary_usd.csv"
)
damage_increase_return_period_summary_readable_file_usd = (
    damage_estimates_directory / "damage_increase_return_period_summary_readable_usd.csv"
)
damage_increase_subsector_summary_file_usd = (
    damage_estimates_directory / "damage_increase_subsector_return_period_usd.csv"
)
damage_increase_subsector_summary_readable_file_usd = (
    damage_estimates_directory / "damage_increase_subsector_return_period_readable_usd.csv"
)

damage_increase_return_period_summary_usd.to_csv(
    damage_increase_return_period_summary_file_usd,
    index=False,
)
damage_increase_return_period_summary_readable_usd.to_csv(
    damage_increase_return_period_summary_readable_file_usd,
    index=False,
)
damage_increase_subsector_summary_usd.to_csv(
    damage_increase_subsector_summary_file_usd,
    index=False,
)
damage_increase_subsector_summary_readable_usd.to_csv(
    damage_increase_subsector_summary_readable_file_usd,
    index=False,
)

print(f"Saved: {damage_increase_return_period_summary_file_usd}")
print(f"Saved: {damage_increase_return_period_summary_readable_file_usd}")
print(f"Saved: {damage_increase_subsector_summary_file_usd}")
print(f"Saved: {damage_increase_subsector_summary_readable_file_usd}")
print()
print("Damage increase totals by return period (USD):")
display(damage_increase_return_period_summary_readable_usd)
print()
print("Damage increase subsectors by return period (USD):")
display(damage_increase_subsector_summary_readable_usd)
